In [ ]:
#pre- trained oil model

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the image dimensions and batch size
img_width, img_height = 224, 224
batch_size = 32
epochs = 10  # Number of training epochs

# Define the paths to your training and validation data directories
train_data_dir = r'C:\Users\User\sample\new_Dataset\train'
validation_data_dir =  r'C:\Users\User\sample\new_Dataset\val'

# Create data generators with data augmentation for training and validation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary'
)

# Define the base model (VGG16)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_width, img_height, 3))

# Freeze the layers in the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers for fine-tuning
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)  # Output layer for binary classification

# Create the transfer learning model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

model.save('newmodel.keras')

# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(validation_generator)
print(f'Test Accuracy: {test_accuracy}')


In [ ]:
#base code with image input - with an optional dry model

import cv2
import numpy as np
from tensorflow.keras.models import load_model 

# Load your pre-trained face oil detection model
oil_model = load_model('newmodel.keras')

#dry_model = load_model('face_dryness_detection_model_transfer_learning.h5')

# Load pre-trained face detection model if needed
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(face_img, target_size=(224, 224)):
    # Resize the image to the target size
    processed_img = cv2.resize(face_img, target_size)
    # Normalize pixel values to the range [0, 1]
    processed_img = processed_img.astype('float32') / 255.0
    # Add batch dimension for compatibility with model input shape
    processed_img = np.expand_dims(processed_img, axis=0)
    return processed_img

def predict_skin_type(image_path):
    # Load the input image
    input_image = cv2.imread(image_path)
    # Preprocess the input image
    processed_image = preprocess_image(input_image)

    # Use the pre-trained model to predict skin type (oily or non-oily)
    oil_prediction = oil_model.predict(processed_image)
    
    #dry_prediction = dry_model.predict(processed_image)
    
    # Interpret the prediction result
    skin_type = 'Oily' if oil_prediction[0][0] > 0.5 else 'Non-oily'
    percentage = oil_prediction[0][0] if skin_type == 'Oily' else 1 - oil_prediction[0][0]
    
    #skin_type = 'Oily' if oil_prediction[0][0] > dry_prediction[0][0] else 'Non-oily'
    #oil_percentage = oil_prediction[0][0] if skin_type == 'Oily' else dry_prediction[0][0]
    #dry_percentage = dry_prediction[0][0]
    
    return skin_type, percentage

    #return skin_type, oil_percentage, dry_percentage

# Example usage:
image_path = r"C:\Users\User\OneDrive\Desktop\TheGoodTheBadTheOily.jpg"
skin_type, percentage = predict_skin_type(image_path)
print(f"Predicted Skin Type: {skin_type} percentage: {percentage:.2f}")

#skin_type, oil_percentage, dry_percentage = predict_skin_type(image_path)
#print(f"Predicted Skin Type: {skin_type}  oil_percentage: {oil_percentage:.2f} %")
#print(f"dry_percentage:{dry_percentage:.2f}%")


In [ ]:
#determining  oil level code

import cv2
import numpy as np
from tensorflow.keras.models import load_model 

# Load your pre-trained face oil detection model
oil_model = load_model('newmodel.keras')

# Load pre-trained face detection model if needed
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(face_img, target_size=(224, 224)):
    # Resize the image to the target size
    processed_img = cv2.resize(face_img, target_size)
    # Normalize pixel values to the range [0, 1]
    processed_img = processed_img.astype('float32') / 255.0
    # Add batch dimension for compatibility with model input shape
    processed_img = np.expand_dims(processed_img, axis=0)
    return processed_img

def map_to_level(oil_prediction):
    if oil_prediction < 0.25:
        return "Low level"
    elif oil_prediction < 0.5:
        return "Normal level"
    elif oil_prediction < 0.75:
        return "Middle level"
    else:
        return "High level"

def predict_skin_type(image_path):
    # Load the input image
    input_image = cv2.imread(image_path)
    # Preprocess the input image
    processed_image = preprocess_image(input_image)

    # Use the pre-trained model to predict skin type (oily or non-oily)
    oil_prediction = oil_model.predict(processed_image)
    
     # Interpret the prediction result
    skin_type = 'Oily' if oil_prediction[0][0] > 0.5 else 'Non-oily'
    percentage = oil_prediction[0][0] if skin_type == 'Oily' else 1 - oil_prediction[0][0]
    
    oiliness_level = map_to_level(oil_prediction[0][0])
    
    return skin_type, percentage, oiliness_level
    
    # Example usage:
image_path = r"C:\Users\User\OneDrive\Desktop\oil.jpg"
#skin_type, percentage = predict_skin_type(image_path)
#print(f"Predicted Skin Type: {skin_type} percentage: {percentage*100:.2f}%")
skin_type, percentage, oiliness_level = predict_skin_type(image_path)
print(f"Predicted Skin Type: {skin_type} | Oiliness Percentage: {percentage*100:.2f}% | Oiliness Level: {oiliness_level}")


In [ ]:
#code with video capture

import cv2
import numpy as np
from tensorflow.keras.models import load_model 

# Load your pre-trained face oil detection model
oil_model = load_model('newmodel.keras')

# Load pre-trained face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(face_img, target_size=(224, 224)):
    # Resize the image to the target size
    processed_img = cv2.resize(face_img, target_size)
    # Normalize pixel values to the range [0, 1]
    processed_img = processed_img.astype('float32') / 255.0
    # Add batch dimension for compatibility with model input shape
    processed_img = np.expand_dims(processed_img, axis=0)
    return processed_img

def map_to_level(oil_prediction):
    if oil_prediction < 0.25:
        return "Low level"
    elif oil_prediction < 0.5:
        return "Normal level"
    elif oil_prediction < 0.75:
        return "Middle level"
    else:
        return "High level"

def predict_skin_type(image_path):
    # Load the input image
    input_image = cv2.imread(image_path)
    # Preprocess the input image
    processed_image = preprocess_image(input_image)

    # Use the pre-trained model to predict skin type (oily or non-oily)
    oil_prediction = oil_model.predict(processed_image)
    
     # Interpret the prediction result
    skin_type = 'Oily' if oil_prediction[0][0] > 0.5 else 'Non-oily'
    percentage = oil_prediction[0][0] if skin_type == 'Oily' else 1 - oil_prediction[0][0]
    
    oiliness_level = map_to_level(oil_prediction[0][0])
    
    return skin_type, percentage, oiliness_level
    
# Initialize the video capture object
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    cv2.imshow('Press Space to Capture Image', frame)
    if cv2.waitKey(1) & 0xFF == ord(' '):  # Wait for space key to be pressed
        # Save the captured frame to a temporary image file
        cv2.imwrite('temp_image.jpg', frame)
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()

# Get the predicted skin type, percentage, and oiliness level for the captured image
image_path = 'temp_image.jpg'
skin_type, percentage, oiliness_level = predict_skin_type(image_path)
print(f"Predicted Skin Type: {skin_type} | Oiliness Percentage: {percentage*100:.2f}% | Oiliness Level: {oiliness_level}")


In [ ]:
#code with skin care treatment according to the skin type

import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load your pre-trained face oil detection model
oil_model = load_model('newmodel.keras')

# Load pre-trained face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(face_img, target_size=(224, 224)):
    # Resize the image to the target size
    processed_img = cv2.resize(face_img, target_size)
    # Normalize pixel values to the range [0, 1]
    processed_img = processed_img.astype('float32') / 255.0
    # Add batch dimension for compatibility with model input shape
    processed_img = np.expand_dims(processed_img, axis=0)
    return processed_img

def map_to_level(oil_prediction):
    if oil_prediction < 0.25:
        return "Low level"
    elif oil_prediction < 0.5:
        return "Normal level"
    elif oil_prediction < 0.75:
        return "Middle level"
    else:
        return "High level"

def predict_skin_type(image_path):
    # Load the input image
    input_image = cv2.imread(image_path)
    # Preprocess the input image
    processed_image = preprocess_image(input_image)

    # Use the pre-trained model to predict skin type (oily or non-oily)
    oil_prediction = oil_model.predict(processed_image)
    
    # Interpret the prediction result
    skin_type = 'Oily' if oil_prediction[0][0] > 0.5 else 'Non-oily'
    percentage = oil_prediction[0][0] if skin_type == 'Oily' else 1 - oil_prediction[0][0]
    
    oiliness_level = map_to_level(oil_prediction[0][0])
    
    return skin_type, percentage, oiliness_level

def recommend_treatment(oiliness_level):
    if oiliness_level == "Low level":
        return ("You have a Balanced Skin ",
                "Use a gentle cleanser suitable for your skin type to maintain balance.",
                "Moisturize regularly to keep skin hydrated and balanced.",
                "Use sunscreen daily to protect against UV damage and premature aging.",
                "Maintain a healthy diet and stay hydrated for overall skin health.")
    elif oiliness_level == "High level":
        return ("You have an Oily Skin ",
                "Use a gentle, foaming cleanser to remove excess oil and impurities.",
                "Use oil-free or mattifying moisturizers to hydrate without adding excess oil.",
                "Use products with ingredients like salicylic acid or benzoyl peroxide to control acne and breakouts.",
                "Use a clay mask or exfoliating treatment 1-2 times a week to help control oil production.",
                "Avoid heavy or greasy products that can clog pores and exacerbate oiliness.")
    elif oiliness_level == "Normal level":
        return ("You have a Dry Skin ",
                "Use a gentle, hydrating cleanser that doesn't strip away natural oils.",
                "Moisturize regularly with a rich, creamy moisturizer to keep skin hydrated.",
                "Use products with ingredients like hyaluronic acid, glycerin, and ceramides to lock in moisture.",
                "Limit hot showers and baths, as hot water can further dry out the skin.",
                "Exfoliate gently to remove dead skin cells and promote cell turnover.",
                "Use a humidifier in dry indoor environments to add moisture to the air.")
    elif oiliness_level == "Middle level":
        return ("You have a Combination Skin ",
                "Use a mild cleanser that doesn't overly dry out or irritate the skin.",
                "Use a lightweight, oil-free moisturizer on areas that tend to be dry.",
                "Use targeted treatments for specific skin concerns, such as acne or dry patches.",
                "Consider using a mattifying primer or oil-absorbing products on oily areas.",
                "Adjust your skincare routine based on how your skin feels in different areas.")
    else:
        return "Skin type not recognized. Please consult a dermatologist for personalized recommendations."

# Initialize the video capture object
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    cv2.imshow('Press Space to Capture Image', frame)
    if cv2.waitKey(1) & 0xFF == ord(' '):  # Wait for space key to be pressed
        # Save the captured frame to a temporary image file
        cv2.imwrite('temp_image.jpg', frame)
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()

# Get the predicted skin type, percentage, and oiliness level for the captured image
image_path = 'temp_image.jpg'
skin_type, percentage, oiliness_level = predict_skin_type(image_path)
print(f"Predicted Skin Type: {skin_type} | Oiliness Percentage: {percentage*100:.2f}% | Oiliness Level: {oiliness_level}")

# Recommend treatment based on oiliness level
treatment_info = recommend_treatment(oiliness_level)
print("\nTreatment Recommendations:")
for info in treatment_info:
    print("- " + info)


In [16]:
#code with recommended beauty products

import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load your pre-trained face oil detection model
oil_model = load_model('newmodel.keras')

# Load pre-trained face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(face_img, target_size=(224, 224)):
    # Resize the image to the target size
    processed_img = cv2.resize(face_img, target_size)
    # Normalize pixel values to the range [0, 1]
    processed_img = processed_img.astype('float32') / 255.0
    # Add batch dimension for compatibility with model input shape
    processed_img = np.expand_dims(processed_img, axis=0)
    return processed_img

def map_to_level(oil_prediction):
    if oil_prediction < 0.25:
        return "Low level"
    elif oil_prediction < 0.5:
        return "Normal level"
    elif oil_prediction < 0.75:
        return "Middle level"
    else:
        return "High level"

def predict_skin_type(image_path):
    # Load the input image
    input_image = cv2.imread(image_path)
    # Preprocess the input image
    processed_image = preprocess_image(input_image)

    # Use the pre-trained model to predict skin type (oily or non-oily)
    oil_prediction = oil_model.predict(processed_image)
    
    # Interpret the prediction result
    skin_type = 'Oily' if oil_prediction[0][0] > 0.5 else 'Non-oily'
    percentage = oil_prediction[0][0] 
    
    oiliness_level = map_to_level(oil_prediction[0][0])
    
    return skin_type, percentage, oiliness_level

def recommend_treatment(oiliness_level):
    if oiliness_level == "Low level":
        return("You have a Dry Skin ",
                "Use a gentle, hydrating cleanser that doesn't strip away natural oils.",
                "Moisturize regularly with a rich, creamy moisturizer to keep skin hydrated.",
                "Use products with ingredients like hyaluronic acid, glycerin, and ceramides to lock in moisture.",
                "Limit hot showers and baths, as hot water can further dry out the skin.",
                "Exfoliate gently to remove dead skin cells and promote cell turnover.",
                "Use a humidifier in dry indoor environments to add moisture to the air.")
    elif oiliness_level == "Normal level":
        return("You have a Balanced Skin ",
                "Use a gentle cleanser suitable for your skin type to maintain balance.",
                "Moisturize regularly to keep skin hydrated and balanced.",
                "Use sunscreen daily to protect against UV damage and premature aging.",
                "Maintain a healthy diet and stay hydrated for overall skin health.")
    elif oiliness_level == "Middle level":
        return("You have a Combination Skin ",
                "Use a mild cleanser that doesn't overly dry out or irritate the skin.",
                "Use a lightweight, oil-free moisturizer on areas that tend to be dry.",
                "Use targeted treatments for specific skin concerns, such as acne or dry patches.",
                "Consider using a mattifying primer or oil-absorbing products on oily areas.",
                "Adjust your skincare routine based on how your skin feels in different areas.")
    elif oiliness_level == "High level":
        return ("You have an Oily Skin ",
                "Use a gentle, foaming cleanser to remove excess oil and impurities.",
                "Use oil-free or mattifying moisturizers to hydrate without adding excess oil.",
                "Use products with ingredients like salicylic acid or benzoyl peroxide to control acne and breakouts.",
                "Use a clay mask or exfoliating treatment 1-2 times a week to help control oil production.",
                "Avoid heavy or greasy products that can clog pores and exacerbate oiliness.") 
    else:
        return "Skin type not recognized. Please consult a dermatologist for personalized recommendations."
    
def recommend_beauty_products(oiliness_level):
    if oiliness_level == "Low level":
        return ("Hydrating Cleanser: CeraVe Hydrating Facial Cleanser",
                    "Rich Moisturizer: Cetaphil Rich Hydrating Night Cream",
                    "Hydrating Serum: Neutrogena Hydro Boost Hydrating Serum")
    elif oiliness_level == "Normal level":
        return ("Gentle Cleanser: Cetaphil Gentle Skin Cleanser",
                    "Moisturizer: Neutrogena Hydro Boost Water Gel",
                    "Sunscreen: La Roche-Posay Anthelios Melt-in Milk Sunscreen")
    elif oiliness_level == "Middle level":
        return ("Mild Cleanser: Cetaphil Daily Facial Cleanser",
                    "Lightweight Moisturizer: La Roche-Posay Toleriane Double Repair Face Moisturizer",
                    "Targeted Treatments: The Ordinary Niacinamide 10% + Zinc 1% Serum for oily areas, The Ordinary Hyaluronic Acid 2% + B5 for dry areas")
    elif oiliness_level == "High level":
        return ("Foaming Cleanser: CeraVe Foaming Facial Cleanser",
                    "Oil-Free Moisturizer: Paula's Choice Skin Balancing Invisible Finish Moisture Gel",
                    "Acne Control Treatment: The Ordinary Salicylic Acid 2% Solution")
    else:
        return "Skin type not recognized. Please consult a dermatologist for personalized recommendations."
    
# Initialize the video capture object
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    cv2.imshow('Press Space to Capture Image', frame)
    if cv2.waitKey(1) & 0xFF == ord(' '):  # Wait for space key to be pressed
        # Save the captured frame to a temporary image file
        cv2.imwrite('temp_image.jpg', frame)
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()

# Get the predicted skin type, percentage, and oiliness level for the captured image
image_path = 'temp_image.jpg'
skin_type, percentage, oiliness_level = predict_skin_type(image_path)
print(f"Predicted Skin Type: {skin_type} | Oiliness Percentage: {percentage*100:.2f}% | Oiliness Level: {oiliness_level}")

# Recommend treatment based on skin type
treatment_info = recommend_treatment(oiliness_level)
print("\nTreatment Recommendations:")
for info in treatment_info:
    print("- " + info)
    
#Recommend beauty products based on skin type
product_info = recommend_beauty_products(oiliness_level)
print("\nBeauty Products Recommendations:")
for info in product_info:
    print("- " + info)

1/1 [==============================] - 0s 217ms/step
Predicted Skin Type: Oily | Oiliness Percentage: 66.24% | Oiliness Level: Middle level

Treatment Recommendations:
- You have a Combination Skin 
- Use a mild cleanser that doesn't overly dry out or irritate the skin.
- Use a lightweight, oil-free moisturizer on areas that tend to be dry.
- Use targeted treatments for specific skin concerns, such as acne or dry patches.
- Consider using a mattifying primer or oil-absorbing products on oily areas.
- Adjust your skincare routine based on how your skin feels in different areas.

Beauty Products Recommendations:
- Mild Cleanser: Cetaphil Daily Facial Cleanser
- Lightweight Moisturizer: La Roche-Posay Toleriane Double Repair Face Moisturizer
- Targeted Treatments: The Ordinary Niacinamide 10% + Zinc 1% Serum for oily areas, The Ordinary Hyaluronic Acid 2% + B5 for dry areas
